In [1]:
import pandas as pd
import numpy as np

import time
import pickle
from pathlib import Path

In [2]:
import black
import jupyter_black

jupyter_black.load(line_length=79)

In [3]:
# FIX VSCODE PATH PROBLEM -- DELETE AFTER
import sys

# add path
sys.path.append(
    "/gpfs01/berens/user/rgonzalesmarquez/phd/llm-excess-vocab/scripts/"
)
print(sys.path)

['/.pyenv/versions/miniconda3-latest/lib/python312.zip', '/.pyenv/versions/miniconda3-latest/lib/python3.12', '/.pyenv/versions/miniconda3-latest/lib/python3.12/lib-dynload', '', '/gpfs01/berens/user/rgonzalesmarquez/.local/lib/python3.12/site-packages', '/gpfs01/berens/user/rgonzalesmarquez/phd/pubmed-landscape', '/gpfs01/berens/user/rgonzalesmarquez/phd/text-embeddings', '/.pyenv/versions/miniconda3-latest/lib/python3.12/site-packages', '/gpfs01/berens/user/rgonzalesmarquez/phd/llm-excess-vocab/scripts/']


In [4]:
%load_ext autoreload
%autoreload 2

from process_pubmed_utils import *

In [5]:
variables_path = Path("../results/variables")
figures_path = Path("../figures")
data_path = Path("../data")
berenslab_data_path = Path("/gpfs01/berens/data/data/pubmed_processed")

In [6]:
print(variables_path)

../results/variables


In [7]:
# MANUAL FIX TO PATH ISSUE FROM VSCODE

nb_path = Path(
    "/gpfs01/berens/user/rgonzalesmarquez/phd/llm-excess-vocab/scripts"
)
assert nb_path.exists(), "The path does not exist"

variables_path = (nb_path / variables_path).resolve(strict=True)
figures_path = (nb_path / figures_path).resolve(strict=True)
data_path = (nb_path / data_path).resolve(strict=True)

We extract from PubMed's metadata `.xml` files the following:
- PubMed ID
- title
- abstract
- language
- journal title
- ISSN
- publication date
- (first and last) author first names
- (first and last) author Affiliations


# Parse data

## 2025 daily updates

In [ ]:
%%time

path = "/gpfs01/berens/data/data/pubmed/2025_daily_updates/"

files_2025_df = import_all_files(path, order_files=True)

# save results
files_2025_df.to_pickle(
    "/gpfs01/berens/data/data/pubmed_processed/files_2025_df_daily_updates_v1"
)

pubmed25n1275.xml
pubmed25n1276.xml
pubmed25n1277.xml
pubmed25n1278.xml
pubmed25n1279.xml
pubmed25n1280.xml
pubmed25n1281.xml
pubmed25n1282.xml
pubmed25n1283.xml
pubmed25n1284.xml
pubmed25n1285.xml
pubmed25n1286.xml
pubmed25n1287.xml
pubmed25n1288.xml
pubmed25n1289.xml
pubmed25n1290.xml
pubmed25n1291.xml
pubmed25n1292.xml
pubmed25n1293.xml
pubmed25n1294.xml
pubmed25n1295.xml
pubmed25n1296.xml
pubmed25n1297.xml
pubmed25n1298.xml
pubmed25n1299.xml
pubmed25n1300.xml
pubmed25n1301.xml
pubmed25n1302.xml
pubmed25n1303.xml
pubmed25n1304.xml
pubmed25n1305.xml
pubmed25n1306.xml
pubmed25n1307.xml
pubmed25n1308.xml
pubmed25n1309.xml
pubmed25n1310.xml
pubmed25n1311.xml
pubmed25n1312.xml
pubmed25n1313.xml
pubmed25n1314.xml
pubmed25n1315.xml
pubmed25n1316.xml
pubmed25n1317.xml
pubmed25n1318.xml
pubmed25n1319.xml
pubmed25n1320.xml
pubmed25n1321.xml
pubmed25n1322.xml
pubmed25n1323.xml
pubmed25n1324.xml
pubmed25n1325.xml
pubmed25n1326.xml
pubmed25n1327.xml
pubmed25n1328.xml
pubmed25n1329.xml
pubmed25n1

In [12]:
print("There are {} papers".format(files_2025_df.shape[0]))

There are 3824653 papers


In [13]:
files_2025_df.head()

,PMID,Title,AbstractText,Language,Journal,Date,NameFirstAuthor,NameLastAuthor,ISSN,AffiliationFirstAuthor,AffiliationLastAuthor,filename
0,10637214,Attorneys argue FDA regulation of tobacco befo...,,eng,Circulation,2000 Jan 18,R,R,1524-4539,,,pubmed25n1275.xml
1,11197357,Brand appearances in contemporary cinema films...,The appearance of a cigarette brand in a cinem...,eng,"Lancet (London, England)",2001 Jan 06,J D,T F,0140-6736,"Department of Pediatrics, Dartmouth Medical Sc...",,pubmed25n1275.xml
2,11226364,Tobacco control in an era of trade liberalisat...,,eng,Tobacco control,2001 Mar,D,I,0964-4563,"Framework Convention on Tobacco Control, Tobac...",,pubmed25n1275.xml
3,11343500,State and federal compliance with the Synar Am...,The Synar Amendment requires states and territ...,eng,Archives of pediatrics & adolescent medicine,2001 May,J R,J R,1072-4710,Department of Family Medicine and Community He...,Department of Family Medicine and Community He...,pubmed25n1275.xml
4,11036748,Tobacco and oral disease. EU-Working Group on ...,,eng,British dental journal,2000 Aug 26,N W,C A,0007-0610,Department of Oral & Maxillofacial Medicine & ...,,pubmed25n1275.xml


In [ ]:
%%time
files_2025_df = pd.read_pickle(
    "/gpfs01/berens/data/data/pubmed_processed/files_2025_df_daily_updates_v1"
)

CPU times: user 48.8 s, sys: 33.3 s, total: 1min 22s
Wall time: 1min 22s


# Filter

Filter out:
- non-English papers
- papers with empty abstracts
- papers with abstracts shorter than 250 or longer than 4000 symbols
- papers with unfinished abstracts

## Empty abstracts and non-english papers

In [14]:
%%time

print("Before, there were {} papers".format(files_2025_df.shape[0]))

# Eliminate empty abstracts
clean_2025_df = files_2025_df[files_2025_df.AbstractText != ""]

print(
    "After eliminating empty abstracts, there are {} papers".format(
        clean_2025_df.shape[0]
    )
)

# Eliminate non-english papers
clean_2025_df = clean_2025_df[clean_2025_df.Language == "eng"]

# print size
print(
    "After first cleaning, there are {} papers".format(clean_2025_df.shape[0])
)

Before, there were 3824653 papers
After eliminating empty abstracts, there are 3502014 papers
After first cleaning, there are 3469558 papers
CPU times: user 3.43 s, sys: 17.9 ms, total: 3.45 s
Wall time: 3.45 s


## Threshold and cut off

### Cut off = 4000

In [15]:
print("Before cut off, there are {} papers".format(clean_2025_df.shape[0]))
abstracts = clean_2025_df["AbstractText"].tolist()
len_strings = map(len, abstracts)
len_abstracts = np.fromiter(len_strings, dtype=np.int64, count=len(abstracts))


cut_off = 4000
clean_2025_df = clean_2025_df[len_abstracts < cut_off]
print("After cut off, there are {} papers".format(clean_2025_df.shape[0]))

Before cut off, there are 3469558 papers
After cut off, there are 3458177 papers


### Threshold = 250

In [16]:
threshold = 250
len_short_abstracts = len_abstracts[len_abstracts < cut_off]
clean_2025_df = clean_2025_df[len_short_abstracts > threshold]
print("After threshold, there are {} papers".format(clean_2025_df.shape[0]))

After threshold, there are 3438550 papers


## Remove the truncated sentence from abstracts


In [17]:
abstracts = clean_2025_df["AbstractText"]
abstracts_list = clean_2025_df["AbstractText"].tolist()

In [18]:
%%time

clean_2025_df.AbstractText = list(
    map(
        lambda x, y: x[: y - 1] if y != -1 else x,
        clean_2025_df.AbstractText,
        clean_2025_df.AbstractText.str.find("ABSTRACT TRUNCATED AT"),
    )
)

CPU times: user 4.18 s, sys: 3.01 ms, total: 4.19 s
Wall time: 4.2 s


## Remove unfinished abstracts

In [19]:
abstracts = clean_2025_df["AbstractText"]
abstracts_list = clean_2025_df["AbstractText"].tolist()
print(len(abstracts_list))

3438550


In [20]:
%%time
end_abstracts = [x[-2:] for x in abstracts_list]

CPU times: user 726 ms, sys: 2.97 ms, total: 728 ms
Wall time: 730 ms


In [21]:
%%time
point_index = np.array([x.find(".") for x in end_abstracts])
question_index = np.array([x.find("?") for x in end_abstracts])
exclamation_index = np.array([x.find("!") for x in end_abstracts])

CPU times: user 1.61 s, sys: 2.01 ms, total: 1.62 s
Wall time: 1.62 s


In [22]:
%%time

print("Before cleaning, there are {} papers".format(clean_2025_df.shape[0]))

# Eliminate unfinished abstracts
clean_2025_df = clean_2025_df[
    (point_index != -1) | (question_index != -1) | (exclamation_index != -1)
]

# print size
print("After cleaning, there are {} papers".format(clean_2025_df.shape[0]))

Before cleaning, there are 3438550 papers
After cleaning, there are 3437773 papers
CPU times: user 1.01 s, sys: 1.98 ms, total: 1.02 s
Wall time: 1.02 s


### Save clean_df

In [24]:
clean_2025_df.head()

,PMID,Title,AbstractText,Language,Journal,Date,NameFirstAuthor,NameLastAuthor,ISSN,AffiliationFirstAuthor,AffiliationLastAuthor,filename
1,11197357,Brand appearances in contemporary cinema films...,The appearance of a cigarette brand in a cinem...,eng,"Lancet (London, England)",2001 Jan 06,J D,T F,0140-6736,"Department of Pediatrics, Dartmouth Medical Sc...",,pubmed25n1275.xml
3,11343500,State and federal compliance with the Synar Am...,The Synar Amendment requires states and territ...,eng,Archives of pediatrics & adolescent medicine,2001 May,J R,J R,1072-4710,Department of Family Medicine and Community He...,Department of Family Medicine and Community He...,pubmed25n1275.xml
5,11338081,The FDA's enforcement of age restrictions on t...,The nation's largest tobacco age-restriction e...,eng,Journal of public health management and practi...,2001 May,S L,M R,1078-4659,"American Legacy Foundation, Washington, DC, USA.",,pubmed25n1275.xml
7,11211641,The cigar revival and the popular press: a con...,The purpose of this study was to examine print...,eng,American journal of public health,2001 Feb,L,L,0090-0036,"Institute for Health Policy Studies, School of...",,pubmed25n1275.xml
10,3948121,Significance of serum protein and lipid-bound ...,A prospective study was done to evaluate the r...,eng,Cancer,1986 Apr 01,K M,J D,0008-543X,,,pubmed25n1275.xml


In [25]:
clean_2025_df.shape

(3437773, 12)

In [26]:
clean_2025_df = clean_2025_df.groupby(["PMID"], as_index=False).last()

In [27]:
clean_2025_df.shape

(1933850, 12)

In [32]:
clean_2025_df.head()

,PMID,Title,AbstractText,Language,Journal,Date,NameFirstAuthor,NameLastAuthor,ISSN,AffiliationFirstAuthor,AffiliationLastAuthor,filename
0,100168,Evaluating cost-effectiveness of diagnostic eq...,An approach to evaluating the cost-effectivene...,eng,British medical journal,1978 Sep 16,J R,D G,0007-1447,,,pubmed25n1420.xml
1,1002,The amino acid sequence of Neurospora NADP-spe...,Peptic and chymotryptic peptides were isolated...,eng,The Biochemical journal,1975 Sep,A A,J R,0264-6021,,,pubmed25n1420.xml
2,1002071,Dilution of blood in fresh water drowning. Pos...,In an attempt to detect signs of dilution of b...,eng,Forensic science,1976 Nov-Dec,L,B,0300-9432,,,pubmed25n1420.xml
3,10021334,Stromal cells mediate retinoid-dependent funct...,The essential role of vitamin A and its metabo...,eng,"Development (Cambridge, England)",1999 Mar,C,J,0950-1991,"Columbia University, Department of Urology, Ne...",,pubmed25n1420.xml
4,10021337,The Drosophila kismet gene is related to chrom...,The Drosophila kismet gene was identified in a...,eng,"Development (Cambridge, England)",1999 Mar,G,J W,0950-1991,"Department of Biology, University of Californi...",,pubmed25n1420.xml


In [33]:
%%time
# save intermediate dataframe

clean_2025_df.to_pickle(berenslab_data_path / "clean_2025_df_daily_updates_v1")

CPU times: user 6.4 s, sys: 883 ms, total: 7.28 s
Wall time: 8.4 s


# Label
Generate labels based on the journal title

In [34]:
label_color_legend = {
    "cancer": "black",
    "neuroscience": "#aeaa00",
    "cardiology": "#1CE6FF",
    "ecology": "#FF34FF",
    "bioinformatics": "#FF4A46",
    "chemistry": "#008941",
    "surgery": "#006FA6",
    "environment": "#0089A3",
    "material": "#0000A6",
    "microbiology": "#B79762",
    "pediatric": "#004D43",
    "immunology": "#8FB0FF",
    "psychology": "#5A0007",
    "psychiatry": "#BA0900",
    "genetics": "#1B4400",
    "nutrition": "#4FC601",
    "veterinary": "#3B5DFF",
    "engineering": "#00C2A0",
    "education": "#549E79",
    "physics": "#BC23FF",
    "optics": "#C895C5",
    "nursing": "#FF2F80",
    "neurology": "#009271",
    "radiology": "#00FECF",
    "ophthalmology": "#A4E804",
    "gynecology": "#FFB500",
    "rehabilitation": "#6B002C",
    "pathology": "#FF9408",
    "anesthesiology": "#CC0744",
    "dermatology": "#D790FF",
    "pharmacology": "#5B4534",
    "physiology": "#E83000",
    "virology": "#6F0062",
    "biochemistry": "#b65141",
    "computation": "#C20078",
    "infectious": "#7A4900",
    "healthcare": "#FF90C9",
    "ethics": "#6508ba",
    "dentistry": "#8e4d8a",
}

In [36]:
%%time
labels_2025, _ = improved_coloring(clean_2025_df.Journal, label_color_legend)

CPU times: user 1min 32s, sys: 50 ms, total: 1min 32s
Wall time: 1min 33s


In [37]:
np.save(variables_path / "labels_2025_du", labels_2025)

# Country
Extract first author's affiliation country

In [38]:
all_countries = [
    "Afghanistan",
    "Albania",
    "Algeria",
    "Andorra",
    "Angola",
    "Antigua and Barbuda",
    "Argentina",
    "Armenia",
    "Australia",
    "Austria",
    "Azerbaijan",
    "Bahamas",
    "Bahrain",
    "Bangladesh",
    "Barbados",
    "Belarus",
    "Belgium",
    "Belize",
    "Benin",
    "Bhutan",
    "Bolivia",
    "Bosnia and Herzegovina",
    "Botswana",
    "Brazil",
    "Brunei",
    "Bulgaria",
    "Burkina Faso",
    "Burundi",
    "Cabo Verde",
    "Cambodia",
    "Cameroon",
    "Canada",
    "Central African Republic",
    "Chad",
    "Chile",
    "China",
    "Colombia",
    "Comoros",
    "Democratic Republic of the Congo",
    "Republic of the Congo",
    "Costa Rica",
    "Côte d’Ivoire",
    "Croatia",
    "Cuba",
    "Cyprus",
    "Czech Republic",
    "Denmark",
    "Djibouti",
    "Dominica",
    "Dominican Republic",
    "East Timor",
    "Ecuador",
    "Egypt",
    "El Salvador",
    "Equatorial Guinea",
    "Eritrea",
    "Estonia",
    "Eswatini",
    "Ethiopia",
    "Fiji",
    "Finland",
    "France",
    "Gabon",
    "Gambia",
    "Georgia",
    "Germany",
    "Ghana",
    "Greece",
    "Grenada",
    "Guatemala",
    "Guinea",
    "Guinea-Bissau",
    "Guyana",
    "Haiti",
    "Honduras",
    "Hungary",
    "Iceland",
    "India",
    "Indonesia",
    "Iran",
    "Iraq",
    "Ireland",
    "Israel",
    "Italy",
    "Jamaica",
    "Japan",
    "Jordan",
    "Kazakhstan",
    "Kenya",
    "Kiribati",
    "North Korea",
    "South Korea",
    "Kosovo",
    "Kuwait",
    "Kyrgyzstan",
    "Laos",
    "Latvia",
    "Lebanon",
    "Nicaragua",
    "Niger",
    "Nigeria",
    "North Macedonia",
    "Norway",
    "Oman",
    "Pakistan",
    "Palau",
    "Panama",
    "Papua New Guinea",
    "Paraguay",
    "Peru",
    "Philippines",
    "Poland",
    "Portugal",
    "Qatar",
    "Romania",
    "Russia",
    "Rwanda",
    "Saint Kitts and Nevis",
    "Saint Lucia",
    "Saint Vincent and the Grenadines",
    "Samoa",
    "San Marino",
    "Sao Tome and Principe",
    "Saudi Arabia",
    "Senegal",
    "Serbia",
    "Seychelles",
    "Sierra Leone",
    "Singapore",
    "Slovakia",
    "Slovenia",
    "Solomon Islands",
    "Somalia",
    "South Africa",
    "Spain",
    "Sri Lanka",
    "Sudan",
    "South Sudan",
    "Suriname",
    "Sweden",
    "Switzerland",
    "Syria",
    "Taiwan",
    "Tajikistan",
    "Tanzania",
    "Thailand",
    "Togo",
    "Tonga",
    "Trinidad and Tobago",
    "Tunisia",
    "Turkey",
    "Turkmenistan",
    "Tuvalu",
    "Uganda",
    "Ukraine",
    "United Arab Emirates",
    "United Kingdom",
    "United States",
    "Uruguay",
    "Uzbekistan",
    "Vanuatu",
    "Vatican City",
    "Venezuela",
    "Vietnam",
    "Yemen",
    "Zambia",
    "Zimbabwe",
]

In [39]:
dict_countries = dict(zip(all_countries, np.arange(1, len(all_countries) + 1)))

## Country mapping

In [40]:
%%time
countries_first_author_2025, _ = mapping_countries(
    clean_2025_df.AffiliationFirstAuthor, dict_countries
)

CPU times: user 7min 4s, sys: 121 ms, total: 7min 4s
Wall time: 7min 5s


## State mapping

In [41]:
all_states = [
    "Alabama",
    "Alaska",
    "Arizona",
    "Arkansas",
    "California",
    "Colorado",
    "Connecticut",
    "Delaware",
    "Florida",
    "Georgia",
    "Hawaii",
    "Idaho",
    "Illinois",
    "Indiana",
    "Iowa",
    "Kansas",
    "Kentucky",
    "Louisiana",
    "Maine",
    "Maryland",
    "Massachusetts",
    "Michigan",
    "Minnesota",
    "Mississippi",
    "Missouri",
    "Montana",
    "Nebraska",
    "Nevada",
    "New Hampshire",
    "New Jersey",
    "New Mexico",
    "New York",
    "North Carolina",
    "North Dakota",
    "Ohio",
    "Oklahoma",
    "Oregon",
    "Pennsylvania",
    "Rhode Island",
    "South Carolina",
    "South Dakota",
    "Tennessee",
    "Texas",
    "Utah",
    "Vermont",
    "Virginia",
    "Washington",
    "West Virginia",
    "Wisconsin",
    "Wyoming",
]

In [42]:
len(all_states)

50

In [43]:
dict_all_states = dict(zip(all_states, np.arange(1, len(all_states) + 1)))

In [44]:
%%time
labels_usa_states_first_author_2025, _ = mapping_states(
    clean_2025_df.AffiliationFirstAuthor, dict_all_states
)

CPU times: user 2min 2s, sys: 53.9 ms, total: 2min 2s
Wall time: 2min 2s


### Correct USA country with states

In [45]:
countries_first_author_2025_usa_corrected = np.where(
    labels_usa_states_first_author_2025 != "unknown",
    "United States",
    countries_first_author_2025,
)

In [46]:
print(
    "Percentage of US papers before correcting",
    np.sum(countries_first_author_2025 == "United States")
    / countries_first_author_2025.shape[0]
    * 100,
)
print(
    "Percentage of US papers after correcting",
    np.sum(countries_first_author_2025_usa_corrected == "United States")
    / countries_first_author_2025.shape[0]
    * 100,
)

Percentage of US papers before correcting 25.810947074488716


Percentage of US papers after correcting 29.672828812989632


In [47]:
np.save(
    variables_path / "countries_first_author_2025_du_usa_corrected",
    countries_first_author_2025_usa_corrected,
)

# Gender

In this part of the notebook we use the `gender` R package. You can install them in R via comannd `install.packages("gender")`.

In [8]:
# load
clean_2025_df = pd.read_pickle(
    berenslab_data_path / "clean_2025_df_daily_updates_v1"
)

## Curate first name lists

### First author

In [9]:
name_first_author = clean_2025_df.NameFirstAuthor

In [10]:
print(
    "Percentge of no names: ",
    np.sum(name_first_author == "") / len(name_first_author) * 100,
)

Percentge of no names:  0.20834087442149082


#### Slicing

In [11]:
%%time
sliced_name_first_author = [
    elem.split()[0] if elem != "" else elem for elem in name_first_author
]

CPU times: user 292 ms, sys: 18.8 ms, total: 310 ms
Wall time: 310 ms


In [12]:
%%time
sliced_name_first_author = [
    elem.split("-")[0] if elem != "" else elem
    for elem in sliced_name_first_author
]

CPU times: user 197 ms, sys: 7.73 ms, total: 204 ms
Wall time: 203 ms


In [13]:
%%time
len_sliced_name_first_author = [len(x) for x in sliced_name_first_author]

CPU times: user 68.5 ms, sys: 938 μs, total: 69.5 ms
Wall time: 68.4 ms


#### Filtering initials

In [14]:
n_initials = len(
    np.array(len_sliced_name_first_author)[
        np.array(len_sliced_name_first_author) == 1
    ]
)
n_total_papers = len(np.array(len_sliced_name_first_author))
print(
    "Percentage of names that are just initials: ",
    n_initials / n_total_papers * 100,
)

Percentage of names that are just initials:  5.935051839594592


In [15]:
print(
    "Percentage of papers with missing names: ",
    np.sum(np.array(sliced_name_first_author) == "")
    / len(sliced_name_first_author)
    * 100,
)

Percentage of papers with missing names:  0.20849600537787316


In [16]:
print(
    "Number of missing names: ",
    len(
        np.array(sliced_name_first_author)[
            np.array(sliced_name_first_author) == ""
        ]
    ),
)

Number of missing names:  4032


In [17]:
%%time
filtered_name_first_author = np.where(
    np.array(len_sliced_name_first_author) == 1, "", sliced_name_first_author
)

CPU times: user 282 ms, sys: 40.6 ms, total: 323 ms
Wall time: 321 ms


In [18]:
n_initials_and_missing = len(
    np.array(filtered_name_first_author)[
        np.array(filtered_name_first_author) == ""
    ]
)
print(
    "Percentage of initials + missing names: ",
    n_initials_and_missing / n_total_papers * 100,
)

Percentage of initials + missing names:  6.143547844972464


### Last author

In [19]:
name_last_author = clean_2025_df.NameLastAuthor

#### Slicing

In [20]:
%%time
sliced_name_last_author = [
    elem.split()[0] if elem != "" else elem for elem in name_last_author
]

CPU times: user 308 ms, sys: 6.27 ms, total: 314 ms
Wall time: 314 ms


In [21]:
%%time
sliced_name_last_author = [
    elem.split("-")[0] if elem != "" else elem
    for elem in sliced_name_last_author
]

CPU times: user 195 ms, sys: 2.98 ms, total: 198 ms
Wall time: 198 ms


In [22]:
%%time
len_sliced_name_last_author = [len(x) for x in sliced_name_last_author]

CPU times: user 66 ms, sys: 2.77 ms, total: 68.8 ms
Wall time: 67.7 ms


#### Filtering initials

In [23]:
n_initials = len(
    np.array(len_sliced_name_last_author)[
        np.array(len_sliced_name_last_author) == 1
    ]
)
n_total_papers = len(np.array(len_sliced_name_last_author))
print(
    "Percentage of names that are just initials: ",
    n_initials / n_total_papers * 100,
)

Percentage of names that are just initials:  6.4153372805543345


In [24]:
print(
    "Number of missing names: ",
    len(
        np.array(sliced_name_last_author)[
            np.array(sliced_name_last_author) == ""
        ]
    ),
)

Number of missing names:  37106


In [25]:
%%time
filtered_name_last_author = np.where(
    np.array(len_sliced_name_last_author) == 1, "", sliced_name_last_author
)

CPU times: user 278 ms, sys: 56.2 ms, total: 334 ms
Wall time: 333 ms


In [26]:
n_initials_and_missing = len(
    np.array(filtered_name_last_author)[
        np.array(filtered_name_last_author) == ""
    ]
)
print(
    "Percentage of initials + missing names: ",
    n_initials_and_missing / n_total_papers * 100,
)

Percentage of initials + missing names:  8.334100369728779


## Gender prediction

In [27]:
%load_ext rpy2.ipython

In [28]:
%R require(gender)
%R require(tibble)

Loading required package: gender
In addition: Warning message:
In (function (package, help, pos = 2, lib.loc = NULL, character.only = FALSE,  :
  library ‘/usr/lib/R/site-library’ contains no packages


Loading required package: tibble


In [29]:
date_year = (
    clean_2025_df.Date.str.extract("([12]\d\d\d)").values.astype(int).squeeze()
)

### First author

In [30]:
# I initialize a dataframe where I will store year, name and gender
# This will be the dataframe `year_name_gender_first_author_df'

df_subset = pd.DataFrame(
    {"Years": date_year, "Names": filtered_name_first_author}
)
df_subset

,Years,Names
0,1978,
1,1975,
2,1976,
3,1999,
4,1999,
...,...,...
1933845,1999,
1933846,1999,
1933847,1999,
1933848,1999,


In [31]:
%%time

# I predict the gender of those names sorting the df by year.
# The problem is that the returned df does not have the same dimensions as the original, since not all the papers have names.
# Therefore, the next two steps are necessary

gender_prediction_df = pd.DataFrame()
unique_years = np.unique(df_subset.Years)

for year in unique_years:
    print("Year: ", year)

    df_grouped_year = df_subset.groupby("Years").get_group(year)
    df_names = df_grouped_year.Names

    # cases for years outside interval
    if year <= 1930:
        %R library(gender)
        %R -i df_names -o result result = gender(df_names, years = 1930, method = "ssa")

    if year >= 2012:
        %R library(gender)
        %R -i df_names -o result result = gender(df_names, years = 2012, method = "ssa")

    if (year < 2012) & (year > 1930):
        %R library(gender)
        %R -i df_names -i year -o result result = gender(df_names, years = year, method = "ssa")

    gender_prediction_df = pd.concat(
        [gender_prediction_df, result], ignore_index=True
    )

gender_prediction_df

Year:  1959
Year:  1960
Year:  1961
Year:  1963
Year:  1966
Year:  1967
Year:  1968
Year:  1969
Year:  1970
Year:  1971
Year:  1972
Year:  1973
Year:  1975
Year:  1976
Year:  1977
Year:  1978
Year:  1979
Year:  1980
Year:  1981
Year:  1982
Year:  1983
Year:  1984
Year:  1985
Year:  1986
Year:  1987
Year:  1988
Year:  1989
Year:  1990
Year:  1991
Year:  1992
Year:  1993
Year:  1994
Year:  1995
Year:  1996
Year:  1997
Year:  1998
Year:  1999
Year:  2000
Year:  2001
Year:  2002
Year:  2003
Year:  2004
Year:  2005
Year:  2006
Year:  2007
Year:  2008
Year:  2009
Year:  2010
Year:  2011
Year:  2012
Year:  2013
Year:  2014
Year:  2015
Year:  2016
Year:  2017
Year:  2018
Year:  2019
Year:  2020
Year:  2021
Year:  2022
Year:  2023
Year:  2024
Year:  2025
Year:  2026
Year:  2027
CPU times: user 31.3 s, sys: 933 ms, total: 32.2 s
Wall time: 32.2 s


,name,proportion_male,proportion_female,gender,year_min,year_max
0,Dennis,0.9952,0.0048,male,1970.0,1970.0
1,Israel,1.0000,0.0000,male,1970.0,1970.0
2,Thomas,0.9940,0.0060,male,1971.0,1971.0
3,Barbara,0.0056,0.9944,female,1979.0,1979.0
4,James,0.9926,0.0074,male,1980.0,1980.0
...,...,...,...,...,...,...
1128380,Yu,0.4553,0.5447,female,2012.0,2012.0
1128381,Yue,0.0000,1.0000,female,2012.0,2012.0
1128382,Yuki,0.2559,0.7441,female,2012.0,2012.0
1128383,Ze,1.0000,0.0000,male,2012.0,2012.0


In [32]:
%%time

# In here I am creating a dictionary that has a map name-gender for every year, based on the predictions from the dataframe above (because they are year dependent).
# I do some tricks to create mappings for also years that were not predicted, and years outside the available `ssa' intervals (<1930, >2012)

unique_predicted_years = np.unique(gender_prediction_df["year_min"])
gender_maps_years = {}
for year in unique_years:
    if year >= 2012:
        eff_year = 2012
    if year <= 1930:
        eff_year = 1930
    if (year < 2012) & (year > 1930):
        eff_year = year

    if eff_year not in unique_predicted_years:
        closest_year = unique_predicted_years[
            (unique_predicted_years - eff_year).argmin()
        ]

        gender_prediction_grouped_year = gender_prediction_df.groupby(
            "year_min"
        ).get_group(closest_year)

    else:
        gender_prediction_grouped_year = gender_prediction_df.groupby(
            "year_min"
        ).get_group(eff_year)

    gender_map = dict(
        zip(
            gender_prediction_grouped_year.name,
            gender_prediction_grouped_year.gender,
        )
    )

    gender_maps_years[year] = gender_map

gender_maps_years.keys()

CPU times: user 3.52 s, sys: 23.8 ms, total: 3.54 s
Wall time: 3.55 s


dict_keys([np.int64(1959), np.int64(1960), np.int64(1961), np.int64(1963), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024

In [33]:
%%time
# Now, using the name-gender maps created above, I map the names in the original dataframe (df_subset) to their respective genders, saving them in a new column 'Gender'

# here I add a 'Gender' column to the df_subset
df_subset["Gender"] = ["unknown"] * df_subset.shape[0]

for year in unique_years:
    print(year)

    df_subset_year = (
        df_subset.groupby("Years")
        .get_group(year)
        .Names.apply(lambda x: np.vectorize(gender_maps_years[year].get)(x))
    )
    df_subset_year.rename("Gender", inplace=True)
    df_subset.update(df_subset_year)

df_subset

1959
1960
1961
1963
1966
1967
1968
1969
1970
1971
1972
1973
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
2027
CPU times: user 25.4 s, sys: 658 ms, total: 26.1 s
Wall time: 26.1 s


,Years,Names,Gender
0,1978,,unknown
1,1975,,unknown
2,1976,,unknown
3,1999,,unknown
4,1999,,unknown
...,...,...,...
1933845,1999,,unknown
1933846,1999,,unknown
1933847,1999,,unknown
1933848,1999,,unknown


In [34]:
# check that the number of predicted genders matches the length of the df with the predictions above
len(df_subset[df_subset.Gender != "unknown"])

1128385

In [35]:
#  save
gender_first_author = df_subset.Gender.to_numpy(dtype=str)
np.save(variables_path / "gender_first_author_2025_du", gender_first_author)

### Last author

In [36]:
df_subset = pd.DataFrame(
    {"Years": date_year, "Names": filtered_name_last_author}
)
df_subset

,Years,Names
0,1978,
1,1975,
2,1976,
3,1999,
4,1999,
...,...,...
1933845,1999,
1933846,1999,
1933847,1999,
1933848,1999,


In [37]:
%%time

gender_prediction_df = pd.DataFrame()
unique_years = np.unique(df_subset.Years)

for year in unique_years:
    print(year)
    df_grouped_year = df_subset.groupby("Years").get_group(year)
    df_names = df_grouped_year.Names

    # cases for years outside interval
    if year <= 1930:
        %R library(gender)
        %R -i df_names -o result result = gender(df_names, years = 1930, method = "ssa")

    if year >= 2012:
        %R library(gender)
        %R -i df_names -o result result = gender(df_names, years = 2012, method = "ssa")

    if (year < 2012) & (year > 1930):
        %R library(gender)
        %R -i df_names -i year -o result result = gender(df_names, years = year, method = "ssa")

    gender_prediction_df = pd.concat(
        [gender_prediction_df, result], ignore_index=True
    )

gender_prediction_df

1959
1960
1961
1963
1966
1967
1968
1969
1970
1971
1972
1973
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
2027
CPU times: user 31 s, sys: 444 ms, total: 31.4 s
Wall time: 31.4 s


,name,proportion_male,proportion_female,gender,year_min,year_max
0,David,0.9955,0.0045,male,1967.0,1967.0
1,Thomas,0.9948,0.0052,male,1967.0,1967.0
2,James,0.9939,0.0061,male,1970.0,1970.0
3,Thomas,0.9948,0.0052,male,1970.0,1970.0
4,Thomas,0.9940,0.0060,male,1971.0,1971.0
...,...,...,...,...,...,...
1153249,Yan,1.0000,0.0000,male,2012.0,2012.0
1153250,Yi,1.0000,0.0000,male,2012.0,2012.0
1153251,Yu,0.4553,0.5447,female,2012.0,2012.0
1153252,Yuan,0.4815,0.5185,female,2012.0,2012.0


In [38]:
%%time

unique_predicted_years = np.unique(gender_prediction_df["year_min"])
gender_maps_years = {}
for year in unique_years:
    if year >= 2012:
        eff_year = 2012
    if year <= 1930:
        eff_year = 1930
    if (year < 2012) & (year > 1930):
        eff_year = year

    if eff_year not in unique_predicted_years:
        closest_year = unique_predicted_years[
            (unique_predicted_years - eff_year).argmin()
        ]

        gender_prediction_grouped_year = gender_prediction_df.groupby(
            "year_min"
        ).get_group(closest_year)

    else:
        gender_prediction_grouped_year = gender_prediction_df.groupby(
            "year_min"
        ).get_group(eff_year)

    gender_map = dict(
        zip(
            gender_prediction_grouped_year.name,
            gender_prediction_grouped_year.gender,
        )
    )

    gender_maps_years[year] = gender_map

gender_maps_years.keys()

CPU times: user 3.56 s, sys: 20.1 ms, total: 3.58 s
Wall time: 3.59 s


dict_keys([np.int64(1959), np.int64(1960), np.int64(1961), np.int64(1963), np.int64(1966), np.int64(1967), np.int64(1968), np.int64(1969), np.int64(1970), np.int64(1971), np.int64(1972), np.int64(1973), np.int64(1975), np.int64(1976), np.int64(1977), np.int64(1978), np.int64(1979), np.int64(1980), np.int64(1981), np.int64(1982), np.int64(1983), np.int64(1984), np.int64(1985), np.int64(1986), np.int64(1987), np.int64(1988), np.int64(1989), np.int64(1990), np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024

In [39]:
%%time

df_subset["Gender"] = ["unknown"] * df_subset.shape[0]

for year in unique_years:
    print(year)

    df_subset_year = (
        df_subset.groupby("Years")
        .get_group(year)
        .Names.apply(lambda x: np.vectorize(gender_maps_years[year].get)(x))
    )
    df_subset_year.rename("Gender", inplace=True)
    df_subset.update(df_subset_year)

df_subset

1959
1960
1961
1963
1966
1967
1968
1969
1970
1971
1972
1973
1975
1976
1977
1978
1979
1980
1981
1982
1983
1984
1985
1986
1987
1988
1989
1990
1991
1992
1993
1994
1995
1996
1997
1998
1999
2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026
2027
CPU times: user 25.6 s, sys: 704 ms, total: 26.3 s
Wall time: 26.3 s


,Years,Names,Gender
0,1978,,unknown
1,1975,,unknown
2,1976,,unknown
3,1999,,unknown
4,1999,,unknown
...,...,...,...
1933845,1999,,unknown
1933846,1999,,unknown
1933847,1999,,unknown
1933848,1999,,unknown


In [40]:
# check that the number of predicted genders matches the length of the df with the predictions above
len(df_subset[df_subset.Gender != "unknown"])

1153254

In [41]:
#  save
gender_last_author = df_subset.Gender.to_numpy(dtype=str)
np.save(variables_path / "gender_last_author_2025_du", gender_last_author)

# Save dataframe


- PMID
- abstract
- journal
- publication year (update also preprocess-and-count bc the year is being extracted there)
- label
- affiliation country (from the first affiliation of the first author).
- inferred gender of first and last author

In [8]:
%%time
clean_2025_df = pd.read_pickle(
    "/gpfs01/berens/data/data/pubmed_processed/clean_2025_df_daily_updates_v1"
)

CPU times: user 4.23 s, sys: 3.06 s, total: 7.29 s
Wall time: 7.4 s


In [10]:
gender_first_author = np.load(
    variables_path / "gender_first_author_2025_du.npy"
)
gender_last_author = np.load(variables_path / "gender_last_author_2025_du.npy")
labels_2025 = np.load(variables_path / "labels_2025_du.npy")
countries_first_author_2025_usa_corrected = np.load(
    variables_path / "countries_first_author_2025_du_usa_corrected.npy"
)

In [11]:
df = clean_2025_df[["PMID", "Title", "AbstractText", "Journal", "Date"]].copy(
    deep=True
)

In [12]:
type(df.PMID.iloc[0])

str

In [13]:
df["Year"] = df.Date.str.extract(r"([12]\d\d\d)").values.astype(int)

In [121]:
months_str = (
    df.Date.str.extract(r"(?:^|\s)([A-Z][a-z]{2})(?=\s|$)")
    .fillna("unknown")
    .to_numpy()
)
months_str = np.array([elem[0] for elem in months_str], dtype="str")

# make sure no other str is extracted
unique_months = [
    "Apr",
    "Aug",
    "Dec",
    "Feb",
    "Jan",
    "Jul",
    "Jun",
    "Mar",
    "May",
    "Nov",
    "Oct",
    "Sep",
    "unknown",
]
assert len(unique_months) == 13
months_str = [
    elem if elem in set(unique_months) else "unknown" for elem in months_str
]

# map to numbers
months_str_to_int_map = {
    "unknown": 0,
    "Jan": 1,
    "Feb": 2,
    "Mar": 3,
    "Apr": 4,
    "May": 5,
    "Jun": 6,
    "Jul": 7,
    "Aug": 8,
    "Sep": 9,
    "Oct": 10,
    "Nov": 11,
    "Dec": 12,
}
df["Month"] = np.vectorize(months_str_to_int_map.get)(months_str)

In [122]:
np.sum(df["Month"] == 0) / len(df["Month"]) * 100

26.028802647568323

In [123]:
df.pop("Date")

0           1978 Sep 16
1              1975 Sep
2          1976 Nov-Dec
3              1999 Mar
4              1999 Mar
               ...     
1933845        1999 Feb
1933846        1999 Feb
1933847        1999 Feb
1933848     1999 Jan 04
1933849     1999 Winter
Name: Date, Length: 1933850, dtype: object

In [124]:
df["Labels"] = labels_2025

In [125]:
df["Countries"] = countries_first_author_2025_usa_corrected

In [126]:
df["InferredGenderFirstAuthor"] = gender_first_author
df["InferredGenderLastAuthor"] = gender_last_author

In [127]:
len(df)

1933850

In [128]:
# in this file this is already done when saving clean_2025_df
df = df.groupby(["PMID"], as_index=False).last()

In [129]:
type(df.PMID.iloc[0])

str

In [130]:
print(f"There are {len(df)} new papers")

There are 1933850 new papers


In [131]:
df.head()

,PMID,Title,AbstractText,Journal,Year,Month,Labels,Countries,InferredGenderFirstAuthor,InferredGenderLastAuthor
0,100168,Evaluating cost-effectiveness of diagnostic eq...,An approach to evaluating the cost-effectivene...,British medical journal,1978,9,unlabeled,unknown,unknown,unknown
1,1002,The amino acid sequence of Neurospora NADP-spe...,Peptic and chymotryptic peptides were isolated...,The Biochemical journal,1975,9,unlabeled,unknown,unknown,unknown
2,1002071,Dilution of blood in fresh water drowning. Pos...,In an attempt to detect signs of dilution of b...,Forensic science,1976,0,unlabeled,unknown,unknown,unknown
3,10021334,Stromal cells mediate retinoid-dependent funct...,The essential role of vitamin A and its metabo...,"Development (Cambridge, England)",1999,3,unlabeled,United States,unknown,unknown
4,10021337,The Drosophila kismet gene is related to chrom...,The Drosophila kismet gene was identified in a...,"Development (Cambridge, England)",1999,3,unlabeled,United States,unknown,unknown


In [132]:
df.shape

(1933850, 10)

In [133]:
%%time
df.to_parquet(
    berenslab_data_path / "pubmed_daily_updates_2025_v1.parquet.gzip",
    index=False,
    engine="pyarrow",
    compression="gzip",
)

CPU times: user 3min 9s, sys: 6.56 s, total: 3min 16s
Wall time: 3min 16s
